# Tuần 2: Thực nghiệm Thiết lập Mốc Baseline (Baseline Benchmark)

**Môn học:** Nhập môn Tối ưu hóa – Trường Đại học Văn Lang  
**Sinh viên:** Đỗ Hoàng Phúc  
**Đề tài:** Cost-Sensitive Machine Learning Optimization for Credit Risk Assessment  

---

## 1. Mục tiêu Tuần 2
1. Chạy mô hình **Random Forest mặc định** (`class_weight=None`, ngưỡng $\tau = 0.5$) qua 5-Fold Stratified CV trên tập Train (800 mẫu).
2. Quét ngưỡng phân loại $\tau \in [0.10, 0.90]$ (bước 0.01) trên xác suất Out-of-Fold (OOF) để tìm ngưỡng tối ưu $\tau^*$ thỏa mãn ràng buộc tỷ lệ từ chối $\le 35\%$.
3. Thu thập bảng số liệu mốc đối chứng (Baseline Table) và vẽ đồ thị **Cost vs. Threshold**.

> **Lưu ý chống rò rỉ dữ liệu (Anti-Leakage):** Toàn bộ thực nghiệm trong notebook này chỉ sử dụng 800 mẫu tập Train. Tập Test 200 mẫu được bảo toàn nguyên vẹn trong kho lạnh.

In [ ]:
import sys
from pathlib import Path

# Thiết lập đường dẫn gốc dự án
PROJECT_ROOT = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / "configs" / "experiment.json").is_file()
     and (folder / "src" / "baseline.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Hãy mở notebook trong thư mục dự án.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

from src.config import load_config
from src.data_utils import load_training_data, load_cv_splits
from src.baseline import main

config = load_config()
print("Đã nạp cấu hình thực nghiệm thành công!")
print(f"- Chi phí FN: {config['cost']['false_negative']} | Chi phí FP: {config['cost']['false_positive']}")
print(f"- Ràng buộc tỷ lệ từ chối tối đa: {config['policy']['max_rejection_rate']:.0%}")

## 2. Nạp dữ liệu huấn luyện và 5-Fold CV Folds

In [ ]:
X_train, y_train = load_training_data(data_dir=PROJECT_ROOT / "data")
cv_splits = load_cv_splits(X_train, data_dir=PROJECT_ROOT / "data")

print(f"Kích thước tập Train: {X_train.shape}")
print(f"Phân bố lớp: Good (0) = {(y_train == 0).sum()} | Bad (1) = {(y_train == 1).sum()}")
print(f"Số lượng CV folds cố định: {len(cv_splits)}")

## 3. Thực thi Baseline mặc định (RF Default, $\tau = 0.5$)

In [ ]:
# Chạy cùng pipeline CLI để kết quả, hình và log luôn thuộc cùng lần chạy.
main()
results_dir = PROJECT_ROOT / "results"
baseline_result = json.loads((results_dir / "baseline_default.json").read_text(encoding="utf-8"))
sweep_result = json.loads((results_dir / "baseline_threshold_sweep.json").read_text(encoding="utf-8"))
bl_metrics = baseline_result["pooled_oof"]
pd.DataFrame([bl_metrics])


## 4. Quét ngưỡng tối ưu trên OOF (Threshold Optimization)

In [ ]:
best_th = sweep_result["best_feasible"]
if best_th is None:
    print("Không có ngưỡng khả thi trong lưới đã xét.")
else:
    display(pd.DataFrame([best_th]))
    saving = (bl_metrics["cost_per_sample"] - best_th["cost_per_sample"]) / bl_metrics["cost_per_sample"] * 100
    print(f"Mức giảm chi phí trên OOF: {saving:.2f}%; chưa xác nhận trên test.")


## 5. Bảng tổng hợp đối sánh kết quả Tuần 2

In [ ]:
comparison = {"RF mặc định": bl_metrics}
if best_th is not None:
    comparison["RF + ngưỡng chọn trên OOF"] = best_th
display(pd.DataFrame(comparison))


## 6. Trực quan hóa Đồ thị và Ma trận nhầm lẫn

In [ ]:
fig_path = PROJECT_ROOT / "figures" / "week2_cost_vs_threshold.png"
if fig_path.exists():
    display(Image(filename=str(fig_path)))
else:
    print("Chưa tìm thấy file đồ thị. Hãy chạy src/baseline.py trước.")

In [ ]:
cm_def = PROJECT_ROOT / "figures" / "week2_cm_baseline_default.png"
cm_opt = PROJECT_ROOT / "figures" / "week2_cm_baseline_threshold_opt.png"

if cm_def.exists() and cm_opt.exists():
    display(Image(filename=str(cm_def)))
    display(Image(filename=str(cm_opt)))

## 7. Diễn giải kết quả & định hướng Tuần 3

- Chi phí dùng ma trận giả định FN:FP = 5:1, tính bằng đơn vị tương đối; không suy ra số tiền ngân hàng tiết kiệm thực tế.
- Ngưỡng được chọn bằng chi phí thấp nhất trong các điểm khả thi của lưới 0.10–0.90, bước 0.01. Nếu hòa chi phí, chọn ngưỡng nhỏ nhất.
- Bảng phía trên là kết quả trên OOF của tập train, cũng là dữ liệu dùng để chọn ngưỡng. Đây chưa phải đánh giá độc lập; giữ test cho đánh giá cuối sau khi chốt cấu hình.
- Giảm FN thường đánh đổi bằng tăng FP. Cần đọc cả chi phí và tỷ lệ từ chối, không chỉ accuracy.
- Tuần 3: so sánh Random Search và Optuna TPE khi tối ưu đồng thời bốn siêu tham số RF và ngưỡng, với cùng ngân sách; chưa khẳng định thuật toán nào sẽ cải thiện.
